In [ ]:
#imports
import os
import glob
import random

import numpy as np
import pandas as pd
from collections import Counter

import cv2
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import transforms, models
from transformers import CLIPTokenizer, CLIPTextModel
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

from torchinfo import summary

from sklearn.model_selection import GroupShuffleSplit
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score
)

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.manifold import TSNE

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False
os.environ["PYTHONHASHSEED"] = str(SEED)

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
BATCH_SIZE = 32

train_tfms = transforms.Compose([
    transforms.Resize(288),
    transforms.RandomResizedCrop(256, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomApply([transforms.ColorJitter(brightness=0.2, contrast=0.2, hue=0.05)], p=0.5),
    transforms.RandomAffine(degrees=180, translate=(0.1, 0.1), scale=(0.9, 1.1), shear=15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_test_tfms = transforms.Compose([
    transforms.Resize(288),
    transforms.CenterCrop(256),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
# 1. Load PAD-UFES-20 metadata
meta_path = "/kaggle/input/datasets/sujansarkar/pad-ufes-20-classification/metadata.csv"
df = pd.read_csv(meta_path)

# 2. Initialize the Group Splitter to prevent patient data leakage
gss_train = GroupShuffleSplit(n_splits=1, train_size=0.7, random_state=42)
train_idx, temp_idx = next(gss_train.split(df, groups=df['patient_id']))

train_df = df.iloc[train_idx].copy()
temp_df = df.iloc[temp_idx].copy()

gss_val = GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=42)
val_idx, test_idx = next(gss_val.split(temp_df, groups=temp_df['patient_id']))

val_df = temp_df.iloc[val_idx].copy()
test_df = temp_df.iloc[test_idx].copy()

print("--- PAD-UFES-20 Data Split Report ---")
print(f"Train Set: {len(train_df):>4} images | Patients: {train_df['patient_id'].nunique()}")
print(f"Val Set:   {len(val_df):>4} images | Patients: {val_df['patient_id'].nunique()}")
print(f"Test Set:  {len(test_df):>4} images | Patients: {test_df['patient_id'].nunique()}")

print("\n--- Class Distribution Check ---")
print("Train:", dict(train_df['diagnostic'].value_counts()))
print("Val:  ", dict(val_df['diagnostic'].value_counts()))
print("Test: ", dict(test_df['diagnostic'].value_counts()))

In [ ]:
def crop_and_save_dataset(input_dir, output_dir, padding=20, min_area_ratio=0.05):
    os.makedirs(output_dir, exist_ok=True)
    
    image_paths = glob.glob(os.path.join(input_dir, '**', '*.png'), recursive=True)
    
    for img_path in tqdm(image_paths, desc="Auto-Cropping Lesions"):
        
        rel_path = os.path.relpath(img_path, input_dir)
        out_path = os.path.join(output_dir, rel_path)
        
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        
        img = cv2.imread(img_path)
        if img is None: continue

        img_area = img.shape[0] * img.shape[1]
            
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        blur = cv2.GaussianBlur(gray, (15, 15), 0)
        
        _, thresh = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        
        kernel = np.ones((9,9), np.uint8)
        closing = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
        
        contours, _ = cv2.findContours(closing, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        if not contours:
            cv2.imwrite(out_path, img)
            continue
            
        largest_contour = max(contours, key=cv2.contourArea)
        
        contour_area = cv2.contourArea(largest_contour)
        if contour_area < (img_area * min_area_ratio) or contour_area > (img_area * 0.95):
            cv2.imwrite(out_path, img)
            continue
        
        x, y, w, h = cv2.boundingRect(largest_contour)
                
        x = max(0, x - padding)
        y = max(0, y - padding)
        w = min(img.shape[1] - x, w + 2*padding)
        h = min(img.shape[0] - y, h + 2*padding)
        
        cropped = img[y:y+h, x:x+w]
        cv2.imwrite(out_path, cropped)

ORIGINAL_DATA = "/kaggle/input/datasets/sujansarkar/pad-ufes-20-classification/images"
CROPPED_DATA = "/kaggle/working/cropped_pad_ufes_20"

print("Starting segmentation crop (preserving folder structure)...")
crop_and_save_dataset(ORIGINAL_DATA, CROPPED_DATA)
print("Cropping complete.")

In [ ]:
# --- 1. Define Features ---
NUMERICAL_COLS = ['age', 'diameter_1', 'diameter_2']
CATEGORICAL_COLS = [
    'gender', 'fitspatrick', 'region', 'elevation', 
    'smoke', 'drink', 'pesticide', 'skin_cancer_history', 'cancer_history',
    'has_piped_water', 'has_sewage_system', 'itch', 'grew', 'hurt', 'changed', 'bleed',
    'background_father', 'background_mother'
]

# Columns needed for text prompts — must be preserved as raw strings
PROMPT_COLS = ['gender', 'region', 'itch', 'bleed', 'hurt', 'grew', 'changed', 'age']

def preprocess_trimodal_data(train_df, val_df, test_df):
    df_train = train_df.copy()
    df_val = val_df.copy()
    df_test = test_df.copy()
    
    # --- PHASE A: Strict Train-Only Statistical Calculations ---
    train_stats = {}
    for col in NUMERICAL_COLS:
        numeric_col = pd.to_numeric(df_train[col], errors='coerce')
        mean_val = numeric_col.mean()
        std_val = numeric_col.std() + 1e-8
        train_stats[col] = {'mean': mean_val, 'std': std_val}

    # --- PHASE B: Apply Stats & Clean All Splits ---
    for df in [df_train, df_val, df_test]:
        # Numerics
        for col in NUMERICAL_COLS:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            df[col] = df[col].fillna(train_stats[col]['mean'])
            df[f"{col}_norm"] = (df[col] - train_stats[col]['mean']) / train_stats[col]['std']
            
        # Categoricals
        for col in CATEGORICAL_COLS:
            if col in ['itch', 'grew', 'hurt', 'changed', 'bleed']:
                df[col] = df[col].fillna('False').astype(str)
            else:
                df[col] = df[col].fillna('unknown').astype(str)

    # FIX: Save raw prompt columns BEFORE one-hot encoding so they survive get_dummies.
    # These are stored with a 'raw_' prefix and will be used by generate_clinical_prompt.
    for df in [df_train, df_val, df_test]:
        for col in PROMPT_COLS:
            if col in df.columns:
                df[f'raw_{col}'] = df[col].astype(str)

    # --- PHASE C: One-Hot Encoding & Alignment ---
    df_train = pd.get_dummies(df_train, columns=CATEGORICAL_COLS, dummy_na=False)
    df_val   = pd.get_dummies(df_val,   columns=CATEGORICAL_COLS, dummy_na=False)
    df_test  = pd.get_dummies(df_test,  columns=CATEGORICAL_COLS, dummy_na=False)

    df_train, df_val  = df_train.align(df_val,  join='left', axis=1, fill_value=0)
    df_train, df_test = df_train.align(df_test, join='left', axis=1, fill_value=0)

    for df in [df_train, df_val, df_test]:
        if 'img_id' in df.columns:
            df.set_index('img_id', inplace=True)

    return df_train, df_val, df_test

# Execute the leak-proof processor
train_df, val_df, test_df = preprocess_trimodal_data(train_df, val_df, test_df)

# Tabular feature cols (one-hot + normalized numerics)
META_COLS = [c for c in train_df.columns if c.startswith(tuple(CATEGORICAL_COLS)) or c.endswith('_norm')]
NUM_META_FEATURES = len(META_COLS)

print(f"Tabular Vector Size: {NUM_META_FEATURES}")
print(f"Raw prompt cols preserved: {[c for c in train_df.columns if c.startswith('raw_')]}")

In [ ]:
class TriModalDataset(Dataset):
    def __init__(self, df, root_dir, meta_cols, transform=None):
        self.df = df
        self.root_dir = root_dir
        self.meta_cols = meta_cols
        self.transform = transform
        
        self.tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
        
        self.image_paths = {}
        for path in glob.glob(os.path.join(root_dir, '**', '*.png'), recursive=True):
            filename = os.path.basename(path) 
            self.image_paths[filename] = path

        self.classes = sorted(df['diagnostic'].unique())
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}

    def generate_clinical_prompt(self, row):
        # FIX: Access raw_ prefixed columns which survive pd.get_dummies.
        # Without this fix, gender/region/symptoms always returned the default
        # value because those columns no longer exist after one-hot encoding.
        age    = str(row.get('raw_age', 'unknown'))
        gender = str(row.get('raw_gender', 'patient')).lower()
        region = str(row.get('raw_region', 'skin')).lower()
        
        symptoms = []
        if str(row.get('raw_itch',    'False')) == 'True': symptoms.append("itches")
        if str(row.get('raw_bleed',   'False')) == 'True': symptoms.append("bleeds")
        if str(row.get('raw_hurt',    'False')) == 'True': symptoms.append("hurts")
        if str(row.get('raw_grew',    'False')) == 'True': symptoms.append("has grown recently")
        if str(row.get('raw_changed', 'False')) == 'True': symptoms.append("has changed in appearance")
        
        symptom_text = " The lesion " + ", ".join(symptoms) + "." if symptoms else ""
        
        prompts = [
            f"A clinical photograph of a skin lesion on the {region} of a {age}-year-old {gender}.{symptom_text}",
            f"Dermatology view of a tumor located on the {region}. Patient is {gender}, age {age}.{symptom_text}",
            f"Macro photography of a skin abnormality. Demographics: {age} yo {gender}. Location: {region}.{symptom_text}"
        ]
        
        return random.choice(prompts)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = self.df.index[idx]
        row = self.df.iloc[idx] 
        
        # Stream 1: Vision
        img_path = self.image_paths[img_id]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
            
        # Stream 2: Language
        text_prompt = self.generate_clinical_prompt(row)
        tokens = self.tokenizer(
            text_prompt, padding='max_length', max_length=77, 
            truncation=True, return_tensors="pt"
        )
        input_ids      = tokens['input_ids'].squeeze(0)
        attention_mask = tokens['attention_mask'].squeeze(0)
        
        # Stream 3: Tabular
        tabular_vector = row[self.meta_cols].values.astype(np.float32)
            
        label = self.class_to_idx[row['diagnostic']]
                
        return image, input_ids, attention_mask, torch.tensor(tabular_vector), label

In [ ]:
DATA_ROOT = "/kaggle/working/cropped_pad_ufes_20"

train_dataset = TriModalDataset(train_df, DATA_ROOT, META_COLS, transform=train_tfms)
val_dataset   = TriModalDataset(val_df,   DATA_ROOT, META_COLS, transform=val_test_tfms)
test_dataset  = TriModalDataset(test_df,  DATA_ROOT, META_COLS, transform=val_test_tfms)

# Class weights for FocalLoss — inverse sqrt-frequency, normalised to mean=1
train_targets    = [train_dataset.class_to_idx[label] for label in train_df['diagnostic']]
class_counts     = Counter(train_targets)
NUM_CLS          = len(class_counts)
total_samples    = sum(class_counts.values())
raw_weights      = [total_samples / (NUM_CLS * class_counts[i] ** 0.5) for i in range(NUM_CLS)]
mean_w           = sum(raw_weights) / NUM_CLS
class_weights_tensor = torch.FloatTensor([w / mean_w for w in raw_weights])

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=4, pin_memory=True, drop_last=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=4, pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=4, pin_memory=True
)

In [ ]:
class_to_idx = train_dataset.class_to_idx
classes = train_dataset.classes

IDX_TO_CLASS = {
    class_to_idx["ACK"]: "Actinic Keratosis (ACK)",
    class_to_idx["BCC"]: "Basal Cell Carcinoma (BCC)",
    class_to_idx["MEL"]: "Melanoma (MEL)",
    class_to_idx["NEV"]: "Nevus (NEV)",
    class_to_idx["SCC"]: "Squamous Cell Carcinoma (SCC)",
    class_to_idx["SEK"]: "Seborrheic Keratosis (SEK)"
}

NUM_CLASSES = len(IDX_TO_CLASS)
class_names = [IDX_TO_CLASS[i] for i in range(NUM_CLASSES)]

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))
print("\nClass to index mapping:", class_to_idx)
print("\nReadable class names:")
for i, name in enumerate(class_names):
    print(i, "\u2192", name)

In [ ]:
class FusionNet(nn.Module):
    def __init__(self, num_meta_features, num_classes=6, pretrained=True, freeze_backbone=False):
        super().__init__()

        # Vision Branch A: ConvNeXt-Tiny
        convnext = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT if pretrained else None)
        self.convnext_features = convnext.features
        self.convnext_dim = 768

        # Vision Branch B: Swin-Tiny
        swin = models.swin_t(weights=models.Swin_T_Weights.DEFAULT if pretrained else None)
        self.swin_features = swin.features
        self.swin_norm = swin.norm
        self.swin_dim = swin.head.in_features  # 768

        # Vision Fusion Bottleneck — 768+768=1536 → 1024
        self.vision_fusion_dim = 1024
        self.vision_fuse = nn.Sequential(
            nn.Linear(self.convnext_dim + self.swin_dim, self.vision_fusion_dim),
            nn.LayerNorm(self.vision_fusion_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3)
        )

        # Text Branch: CLIP
        self.text_encoder = CLIPTextModel.from_pretrained("openai/clip-vit-base-patch32")
        self.text_dim = self.text_encoder.config.hidden_size  # 512

        # Shared projection space
        self.embed_dim = 256
        self.vis_proj  = nn.Linear(self.vision_fusion_dim, self.embed_dim)
        self.text_proj = nn.Linear(self.text_dim, self.embed_dim)

        # Learnable positional encoding for the 8x8=64 vision patch grid
        self.pos_embed = nn.Parameter(torch.randn(1, 64, self.embed_dim) * 0.02)

        # Bidirectional cross-attention
        self.text_to_vis_attn = nn.MultiheadAttention(
            embed_dim=self.embed_dim, num_heads=8, dropout=0.1, batch_first=True
        )
        self.vis_to_text_attn = nn.MultiheadAttention(
            embed_dim=self.embed_dim, num_heads=8, dropout=0.1, batch_first=True
        )

        # Clinical Branch: Tabular MLP
        self.tabular_mlp = nn.Sequential(
            nn.Linear(num_meta_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3)
        )
        self.tabular_dim = 128
        self.meta_proj = nn.Linear(self.tabular_dim, self.embed_dim)

        # Joint multimodal gate — sees all 5 streams concatenated so it can
        # make relative comparisons between modalities per sample
        self.num_modalities = 5
        self.modal_gate = nn.Linear(self.embed_dim * self.num_modalities, self.num_modalities)

        # Final Classification Head
        self.classifier = nn.Sequential(
            nn.Linear(self.embed_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

        if freeze_backbone:
            for p in self.convnext_features.parameters(): p.requires_grad = False
            for p in self.swin_features.parameters():     p.requires_grad = False
            for p in self.swin_norm.parameters():         p.requires_grad = False
            for p in self.text_encoder.parameters():      p.requires_grad = False

    def forward(self, x, input_ids, attention_mask, meta_data, return_attn=False):
        # Vision
        f_convnext = self.convnext_features(x).flatten(2).transpose(1, 2)  # [B, 64, 768]
        f_swin     = self.swin_norm(self.swin_features(x)).flatten(1, 2)   # [B, 64, 768]
        fused_vision = self.vision_fuse(torch.cat([f_convnext, f_swin], dim=-1))  # [B, 64, 1024]

        # Text
        text_feats = self.text_encoder(
            input_ids=input_ids, attention_mask=attention_mask
        ).last_hidden_state  # [B, 77, 512]

        # Project to shared space
        v_emb = F.normalize(self.vis_proj(fused_vision) + self.pos_embed, p=2, dim=-1)  # [B, 64, 256]
        t_emb = F.normalize(self.text_proj(text_feats),                   p=2, dim=-1)  # [B, 77, 256]

        # Bidirectional cross-attention — names reflect which stream is the query
        # text_to_vis: query=text → output has text sequence length [B, 77, 256]
        t_attended, attn_weights = self.text_to_vis_attn(query=t_emb, key=v_emb, value=v_emb)
        # vis_to_text: query=vision → output has vision sequence length [B, 64, 256]
        v_attended, _            = self.vis_to_text_attn(query=v_emb, key=t_emb, value=t_emb)

        t_attended_pooled = t_attended.mean(dim=1)  # [B, 256] — text enriched with visual context
        v_attended_pooled = v_attended.mean(dim=1)  # [B, 256] — vision enriched with text context
        global_text       = t_emb.mean(dim=1)       # [B, 256]
        global_vision     = v_emb.mean(dim=1)       # [B, 256]

        # Tabular
        f_meta      = self.tabular_mlp(meta_data)
        f_meta_proj = self.meta_proj(f_meta)         # [B, 256]

        # Joint gate — concatenate all streams so gate can compare them against each other
        modalities   = [t_attended_pooled, v_attended_pooled, global_text, global_vision, f_meta_proj]
        stacked      = torch.stack(modalities, dim=1)                          # [B, 5, 256]
        gate_weights = self.modal_gate(
            torch.cat(modalities, dim=-1)                                      # [B, 1280]
        ).softmax(dim=-1).unsqueeze(-1)                                        # [B, 5, 1]
        fused        = (stacked * gate_weights).sum(dim=1)                     # [B, 256]

        preds = self.classifier(fused)

        if return_attn:
            return preds, attn_weights
        return preds


model = FusionNet(
    num_meta_features=NUM_META_FEATURES,
    num_classes=NUM_CLASSES,
    pretrained=True,
    freeze_backbone=True
).to(DEVICE)

summary(
    model,
    input_size=[
        (BATCH_SIZE, 3, 256, 256),
        (BATCH_SIZE, 77),
        (BATCH_SIZE, 77),
        (BATCH_SIZE, NUM_META_FEATURES)
    ],
    dtypes=[torch.float32, torch.long, torch.long, torch.float32],
    col_names=["input_size", "output_size", "num_params"],
    depth=3
)

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0):
        super().__init__()
        self.gamma = gamma
        # label_smoothing removed — it softens targets in the opposite direction
        # to focal loss's hard-example focus, partially cancelling the effect
        self.ce = nn.CrossEntropyLoss(weight=weight, reduction="none")

    def forward(self, inputs, targets):
        ce_loss = self.ce(inputs, targets)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

In [ ]:
criterion = FocalLoss(weight=class_weights_tensor.to(DEVICE), gamma=2.0)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4,
    weight_decay=1e-2
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=10,
    T_mult=2,
    eta_min=1e-6
)

In [ ]:
def compute_specificity(y_true, y_pred, num_classes):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    specificities = []
    for i in range(num_classes):
        TP = cm[i, i]
        FP = cm[:, i].sum() - TP
        FN = cm[i, :].sum() - TP
        TN = cm.sum() - (TP + FP + FN)
        specificities.append(TN / (TN + FP) if (TN + FP) > 0 else 0.0)
    return float(np.mean(specificities))

In [ ]:
scaler = GradScaler('cuda')

def mixup_batch(images, meta_data, labels, alpha=0.3):
    lam      = np.random.beta(alpha, alpha)
    rand_idx = torch.randperm(images.size(0), device=images.device)
    # mix both image and tabular with the same lam and permutation index
    mixed_images = lam * images    + (1 - lam) * images[rand_idx]
    mixed_meta   = lam * meta_data + (1 - lam) * meta_data[rand_idx]
    # text tokens can't be linearly interpolated, so input_ids/mask stay from A;
    # we accept this approximation and still mix labels for image+tabular signal
    return mixed_images, mixed_meta, labels, labels[rand_idx], lam

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    y_true, y_pred = [], []

    for images, input_ids, attention_mask, meta_data, labels in tqdm(loader, desc="Training"):
        images         = images.to(device)
        input_ids      = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        meta_data      = meta_data.to(device)
        labels         = labels.to(device)

        mixed_images, mixed_meta, y_a, y_b, lam = mixup_batch(images, meta_data, labels)

        optimizer.zero_grad()

        with autocast('cuda'):
            outputs = model(mixed_images, input_ids, attention_mask, mixed_meta)
            loss    = lam * criterion(outputs, y_a) + (1 - lam) * criterion(outputs, y_b)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(outputs, 1)
        y_true.extend(y_a.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

    avg_loss = running_loss / len(y_true)
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec  = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1   = f1_score(y_true, y_pred, average="macro", zero_division=0)
    spec = compute_specificity(y_true, y_pred, NUM_CLASSES)
    return avg_loss, acc, prec, rec, f1, spec

In [ ]:
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    y_true, y_pred = [], []

    with torch.no_grad():
        for images, input_ids, attention_mask, meta_data, labels in tqdm(loader, desc="Validation", leave=False):
            images         = images.to(device)
            input_ids      = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            meta_data      = meta_data.to(device)
            labels         = labels.to(device)
            
            outputs = model(images, input_ids, attention_mask, meta_data)
            loss    = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            preds = torch.argmax(outputs, 1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    avg_loss = running_loss / len(y_true)
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec  = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1   = f1_score(y_true, y_pred, average="macro", zero_division=0)
    spec = compute_specificity(y_true, y_pred, NUM_CLASSES)
    return avg_loss, acc, prec, rec, f1, spec, y_true, y_pred

In [ ]:
PHASE1_EPOCHS = 2

train_losses, val_losses = [], []
best_val_f1 = 0.0
best_model_state = None

for epoch in range(PHASE1_EPOCHS):
    print(f"\nPhase 1 Epoch [{epoch+1}/{PHASE1_EPOCHS}]")

    train_loss, train_acc, train_prec, train_rec, train_f1, train_spec = train_one_epoch(
        model, train_loader, optimizer, criterion, DEVICE
    )
    val_loss, val_acc, val_prec, val_rec, val_f1, val_spec, _, _ = evaluate(
        model, val_loader, criterion, DEVICE
    )

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    scheduler.step()

    print(f"LR: {optimizer.param_groups[0]['lr']:.2e}")
    print(f"Train | Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | Prec: {train_prec:.4f} | Rec: {train_rec:.4f} | F1: {train_f1:.4f} | Spec: {train_spec:.4f}")
    print(f"Val   | Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | Prec: {val_prec:.4f} | Rec: {val_rec:.4f} | F1: {val_f1:.4f} | Spec: {val_spec:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model_state = model.state_dict()
        print(f"Validation F1 improved ({val_f1:.4f}) — saving best Phase 1 model")

if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print("\nPhase 1 complete. Best model loaded.")

In [ ]:
for param in model.parameters():
    param.requires_grad = True

In [ ]:
SWIN_LR     = 1e-6
CLIP_LR     = 1e-6
CONVNEXT_LR = 1e-5
HEAD_LR     = 1e-4

optimizer = torch.optim.AdamW([
    {'params': model.swin_features.parameters(),     'lr': SWIN_LR},
    {'params': model.swin_norm.parameters(),         'lr': SWIN_LR},
    {'params': model.text_encoder.parameters(),      'lr': CLIP_LR},
    {'params': model.convnext_features.parameters(), 'lr': CONVNEXT_LR},

    {'params': model.vision_fuse.parameters(),       'lr': HEAD_LR},
    {'params': model.vis_proj.parameters(),          'lr': HEAD_LR},
    {'params': model.text_proj.parameters(),         'lr': HEAD_LR},
    {'params': [model.pos_embed],                    'lr': HEAD_LR},
    {'params': model.text_to_vis_attn.parameters(),  'lr': HEAD_LR},
    {'params': model.vis_to_text_attn.parameters(),  'lr': HEAD_LR},
    {'params': model.tabular_mlp.parameters(),       'lr': HEAD_LR},
    {'params': model.meta_proj.parameters(),         'lr': HEAD_LR},
    {'params': model.modal_gate.parameters(),        'lr': HEAD_LR},
    {'params': model.classifier.parameters(),        'lr': HEAD_LR},
], weight_decay=5e-2)

scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=10,
    T_mult=2,
    eta_min=1e-7
)

In [ ]:
PHASE2_EPOCHS = 60

fine_train_losses, fine_val_losses = [], []
best_model_state = None

patience = 15
counter  = 0

for epoch in range(PHASE2_EPOCHS):
    print(f"\nPhase 2 Epoch [{epoch+1}/{PHASE2_EPOCHS}]")

    train_loss, train_acc, train_prec, train_rec, train_f1, train_spec = train_one_epoch(
        model, train_loader, optimizer, criterion, DEVICE
    )
    val_loss, val_acc, val_prec, val_rec, val_f1, val_spec, _, _ = evaluate(
        model, val_loader, criterion, DEVICE
    )

    fine_train_losses.append(train_loss)
    fine_val_losses.append(val_loss)
    scheduler.step()

    print(f"LR (backbone): {optimizer.param_groups[0]['lr']:.2e} | LR (head): {optimizer.param_groups[-1]['lr']:.2e}")
    print(f"Train | Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | Prec: {train_prec:.4f} | Rec: {train_rec:.4f} | F1: {train_f1:.4f} | Spec: {train_spec:.4f}")
    print(f"Val   | Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | Prec: {val_prec:.4f} | Rec: {val_rec:.4f} | F1: {val_f1:.4f} | Spec: {val_spec:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model_state = model.state_dict()
        torch.save(best_model_state, '/kaggle/working/FusionNet_PAD_Best.pth')
        counter = 0
        print(f"Validation F1 improved ({val_f1:.4f}) — saving best Phase 2 model")
    else:
        counter += 1
        print(f"No improvement. Early stopping counter: {counter}/{patience}")

    if counter >= patience:
        print(f"\nEarly stopping triggered after {patience} epochs without F1 improvement.")
        break

if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print("\nPhase 2 complete. Best model loaded.")

In [ ]:
# TRAIN REPORT

train_loss, train_acc, _, _, _, train_spec, train_true, train_pred = evaluate(
    model, train_loader, criterion, DEVICE
)

print("\nTRAIN REPORT")
print("Train Loss:", f"{train_loss:.4f}")
print("Train Accuracy:", f"{train_acc:.4f}")
print("Train Specificity (Macro):", f"{train_spec:.4f}")
print("\nTrain Classification Report:\n")
print(classification_report(train_true, train_pred, target_names=class_names, zero_division=0, digits=4))

train_macro_f1 = f1_score(train_true, train_pred, average="macro", zero_division=0)
print(f"Train Macro F1: {train_macro_f1:.4f}")

In [ ]:
# VALIDATION REPORT

val_loss, val_acc, _, _, _, val_spec, val_true, val_pred = evaluate(
    model, val_loader, criterion, DEVICE
)

print("\nVALIDATION REPORT")
print("Validation Loss:", f"{val_loss:.4f}")
print("Validation Accuracy:", f"{val_acc:.4f}")
print("Validation Specificity (Macro):", f"{val_spec:.4f}")
print("\nValidation Classification Report:\n")
print(classification_report(val_true, val_pred, target_names=class_names, zero_division=0, digits=4))

val_macro_prec = precision_score(val_true, val_pred, average="macro", zero_division=0)
val_macro_rec  = recall_score(val_true, val_pred, average="macro", zero_division=0)
val_macro_f1   = f1_score(val_true, val_pred, average="macro", zero_division=0)
print(f"Macro Avg -> Precision: {val_macro_prec:.4f} | Recall: {val_macro_rec:.4f} | F1: {val_macro_f1:.4f}")
print(f"Specificity -> {val_spec:.4f}")

In [ ]:
# TEST REPORT

test_loss, test_acc, _, _, _, test_spec, y_true, y_pred = evaluate(
    model, test_loader, criterion, DEVICE
)

print("\nTEST REPORT")
print("Test Loss:", f"{test_loss:.4f}")
print("Test Accuracy:", f"{test_acc:.4f}")
print("Test Specificity (Macro):", f"{test_spec:.4f}")
print("\nTest Classification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0, digits=4))

test_macro_prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
test_macro_rec  = recall_score(y_true, y_pred, average="macro", zero_division=0)
test_macro_f1   = f1_score(y_true, y_pred, average="macro", zero_division=0)

test_weighted_prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
test_weighted_rec  = recall_score(y_true, y_pred, average="weighted", zero_division=0)
test_weighted_f1   = f1_score(y_true, y_pred, average="weighted", zero_division=0)

print("Test Additional Summary:")
print(f"Macro Avg     -> Precision: {test_macro_prec:.4f} | Recall: {test_macro_rec:.4f} | F1: {test_macro_f1:.4f}")
print(f"Weighted Avg  -> Precision: {test_weighted_prec:.4f} | Recall: {test_weighted_rec:.4f} | F1: {test_weighted_f1:.4f}")
print(f"Specificity   -> {test_spec:.4f}")


# TEST CONFUSION MATRIX

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="mako_r", vmax=100,
    xticklabels=class_names, yticklabels=class_names,
    square=True, cbar_kws={"shrink": 0.8}
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("PAD-UFES-20 Test Confusion Matrix")
plt.tight_layout()
plt.show()


# LOSS CURVES

all_train_losses = train_losses + fine_train_losses
all_val_losses   = val_losses   + fine_val_losses

total_epochs = len(all_train_losses)
epochs = range(1, total_epochs + 1)
phase1_end = len(train_losses)

plt.figure(figsize=(10, 5))
plt.plot(epochs, all_train_losses, label="Training Loss")
plt.plot(epochs, all_val_losses,   label="Validation Loss")
plt.axvline(phase1_end, color="gray", linestyle="--", linewidth=0.9, label=f"Phase 1 End (Ep {phase1_end})")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

best_epoch = all_val_losses.index(min(all_val_losses)) + 1

plt.figure(figsize=(10, 5))
plt.plot(epochs, all_val_losses, label="Validation Loss")
plt.axvline(phase1_end, color="gray",  linestyle="--", linewidth=0.9, label=f"Phase 1 End (Ep {phase1_end})")
plt.axvline(best_epoch, color="green", linestyle="-",  linewidth=1.2, label=f"Min Loss Epoch ({best_epoch})")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Validation Loss (green = min loss epoch, model saved on best F1)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
from scipy import stats as scipy_stats

N_BOOTSTRAP = 1000
rng         = np.random.default_rng(42)
y_true_arr  = np.array(y_true)
y_pred_arr  = np.array(y_pred)
n_test      = len(y_true_arr)

boot = {"Accuracy": [], "F1 (Macro)": [], "Precision (Macro)": [], "Recall (Macro)": [], "Specificity": []}
per_class_f1 = {name: [] for name in class_names}

for _ in range(N_BOOTSTRAP):
    idx = rng.integers(0, n_test, size=n_test)
    y_t, y_p = y_true_arr[idx], y_pred_arr[idx]
    boot["Accuracy"].append(accuracy_score(y_t, y_p))
    boot["F1 (Macro)"].append(f1_score(y_t, y_p, average="macro", zero_division=0))
    boot["Precision (Macro)"].append(precision_score(y_t, y_p, average="macro", zero_division=0))
    boot["Recall (Macro)"].append(recall_score(y_t, y_p, average="macro", zero_division=0))
    boot["Specificity"].append(compute_specificity(y_t, y_p, NUM_CLASSES))
    pc = f1_score(y_t, y_p, average=None, zero_division=0, labels=list(range(NUM_CLASSES)))
    for ci, name in enumerate(class_names):
        per_class_f1[name].append(pc[ci])

def ci95(arr): return float(np.percentile(arr, 2.5)), float(np.percentile(arr, 97.5))

print("\n" + "=" * 72)
print(f"{'Metric':<25} {'Mean':>8}  {'Std':>8}  {'95% CI Lower':>14}  {'95% CI Upper':>14}")
print("=" * 72)
for metric, values in boot.items():
    arr = np.array(values); lo, hi = ci95(arr)
    print(f"{metric:<25} {arr.mean():>8.4f}  {arr.std():>8.4f}  {lo:>14.4f}  {hi:>14.4f}")
print("-" * 72)
print(f"\n{'Class':<52} {'Mean':>8}  {'Std':>8}  {'95% CI Lower':>14}  {'95% CI Upper':>14}")
print("-" * 72)
for name in class_names:
    arr = np.array(per_class_f1[name]); lo, hi = ci95(arr)
    print(f"{name:<52} {arr.mean():>8.4f}  {arr.std():>8.4f}  {lo:>14.4f}  {hi:>14.4f}")
print("=" * 72)

fig, axes = plt.subplots(1, 5, figsize=(18, 6))
fig.subplots_adjust(top=0.82, wspace=0.4)
for ax, (label, values) in zip(axes, boot.items()):
    arr = np.array(values); lo, hi = ci95(arr); mn, sd = arr.mean(), arr.std()
    ax.boxplot(arr, widths=0.5, patch_artist=True,
               boxprops=dict(facecolor="#4c72b0", alpha=0.75),
               medianprops=dict(color="white", linewidth=2),
               whiskerprops=dict(linewidth=1.2), capprops=dict(linewidth=1.2),
               flierprops=dict(marker="o", markersize=3, markerfacecolor="gray", alpha=0.5))
    ax.set_title(label, fontsize=10, fontweight="bold", pad=8)
    ax.set_xticks([]); ax.set_ylabel("Score", fontsize=9)
    ax.grid(axis="y", alpha=0.3, linestyle="--")
    ax.text(1, ax.get_ylim()[1], f"μ={mn:.4f}\nσ={sd:.4f}\n95%CI [{lo:.4f},{hi:.4f}]",
            ha="center", va="top", fontsize=7.5, color="#222222",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", edgecolor="gray", alpha=0.8))
plt.suptitle(f"PAD-UFES-20 — Bootstrap Metric Distributions (n={N_BOOTSTRAP}) — Test Set",
             fontsize=13, fontweight="bold", y=0.97)
plt.savefig("/kaggle/working/pad_bootstrap_metric_boxplots.png", dpi=150, bbox_inches="tight")
plt.show()

means  = np.array([np.mean(per_class_f1[n]) for n in class_names])
stds   = np.array([np.std(per_class_f1[n])  for n in class_names])
lo_arr = np.array([ci95(per_class_f1[n])[0] for n in class_names])
hi_arr = np.array([ci95(per_class_f1[n])[1] for n in class_names])

fig, ax = plt.subplots(figsize=(13, 6))
x = np.arange(NUM_CLASSES)
bars = ax.bar(x, means, yerr=[means - lo_arr, hi_arr - means], capsize=6,
              color="#4c72b0", alpha=0.8, ecolor="#333333",
              error_kw={"linewidth": 1.8, "capthick": 1.8}, width=0.55)
ax.set_xticks(x)
ax.set_xticklabels([n.split("(")[-1].replace(")", "").strip() if "(" in n else n
                    for n in class_names], fontsize=11)
ax.set_ylabel("F1 Score", fontsize=11); ax.set_ylim(0, 1.15)
ax.set_title("PAD-UFES-20 — Per-Class F1 Mean ± 95% CI (Bootstrap)", fontsize=13, fontweight="bold")
ax.grid(axis="y", alpha=0.3, linestyle="--"); ax.spines[["top", "right"]].set_visible(False)
for bar, m, s, hi in zip(bars, means, stds, hi_arr):
    ax.text(bar.get_x() + bar.get_width() / 2, hi + 0.025,
            f"{m:.3f}\n±{s:.3f}", ha="center", va="bottom", fontsize=9, color="#111111")
plt.tight_layout()
plt.savefig("/kaggle/working/pad_bootstrap_perclass_f1.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
def evaluate_and_plot_roc(model, loader, class_names):
    model.eval()
    all_targets, all_probs = [], []
    
    print("Extracting Softmax probabilities for ROC Curve...")
    with torch.no_grad():
        for images, input_ids, attention_mask, meta_data, labels in loader:
            images         = images.to(DEVICE)
            input_ids      = input_ids.to(DEVICE)
            attention_mask = attention_mask.to(DEVICE)
            meta_data      = meta_data.to(DEVICE)
            
            outputs = model(images, input_ids, attention_mask, meta_data)
            probs   = F.softmax(outputs, dim=1)
            
            all_targets.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            
    all_targets = np.array(all_targets)
    all_probs   = np.array(all_probs)
    
    y_true_bin = label_binarize(all_targets, classes=range(len(class_names)))
    
    plt.figure(figsize=(10, 8))
    for i in range(len(class_names)):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], all_probs[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, lw=2, label=f'{class_names[i]} (AUC = {roc_auc:.3f})')

    plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Guessing')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title('PAD-UFES-20 Tri-Modal Multi-Class ROC Curve', fontsize=16, fontweight='bold')
    plt.legend(loc="lower right", fontsize=10)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

evaluate_and_plot_roc(model, test_loader, class_names)

In [ ]:
def evaluate_with_tta_and_thresholds(model, loader, device, thresholds):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for images, input_ids, attention_mask, meta_data, labels in tqdm(loader, desc="TTA Testing"):
            images         = images.to(device)
            input_ids      = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            meta_data      = meta_data.to(device)
            
            out_orig  = F.softmax(model(images,                           input_ids, attention_mask, meta_data), dim=1)
            out_hf    = F.softmax(model(torch.flip(images, [3]),          input_ids, attention_mask, meta_data), dim=1)
            out_vf    = F.softmax(model(torch.flip(images, [2]),          input_ids, attention_mask, meta_data), dim=1)
            out_r90   = F.softmax(model(torch.rot90(images, 1, [2, 3]),   input_ids, attention_mask, meta_data), dim=1)
            out_r270  = F.softmax(model(torch.rot90(images, 3, [2, 3]),   input_ids, attention_mask, meta_data), dim=1)
            
            avg_probs      = (out_orig + out_hf + out_vf + out_r90 + out_r270) / 5.0
            adjusted_probs = avg_probs / thresholds.to(device)
            preds          = torch.argmax(adjusted_probs, dim=1)

            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec  = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1   = f1_score(y_true, y_pred, average="macro", zero_division=0)
    spec = compute_specificity(y_true, y_pred, NUM_CLASSES)
    return acc, prec, rec, f1, spec, y_true, y_pred

In [ ]:
# --- PAD-UFES-20 Custom Thresholds ---
custom_thresholds = torch.ones(NUM_CLASSES).to(DEVICE)

custom_thresholds[class_to_idx["MEL"]] = 0.75
custom_thresholds[class_to_idx["SCC"]] = 0.80
custom_thresholds[class_to_idx["ACK"]] = 0.85
custom_thresholds[class_to_idx["BCC"]] = 1.0
custom_thresholds[class_to_idx["SEK"]] = 1.0
custom_thresholds[class_to_idx["NEV"]] = 1.0

tta_acc, tta_prec, tta_rec, tta_f1, tta_spec, tta_true, tta_pred = evaluate_with_tta_and_thresholds(
    model, test_loader, DEVICE, custom_thresholds
)

print("\n--- FINAL TEST REPORT (TRI-MODAL FUSIONNET + TTA + PAD-UFES-20 THRESHOLDS) ---")
print("Test Accuracy:", f"{tta_acc:.4f}")
print("Test Macro F1:", f"{tta_f1:.4f}")

print("\nTest Classification Report:\n")
print(classification_report(tta_true, tta_pred, target_names=class_names, zero_division=0, digits=4))

cm_tta = confusion_matrix(tta_true, tta_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_tta, annot=True, fmt="d", cmap="magma_r", vmax=100,
    xticklabels=class_names, yticklabels=class_names,
    square=True, cbar_kws={"shrink": 0.8}
)
plt.xlabel("Predicted (Refined Thresholds)")
plt.ylabel("Actual")
plt.title("PAD-UFES-20 Test Confusion Matrix (Tri-Modal + TTA)")
plt.tight_layout()
plt.show()

In [ ]:
def visualize_attention_random(model, loader, class_names, num_images=10):
    model.eval()
    
    all_images, all_input_ids, all_masks, all_meta, all_labels = [], [], [], [], []

    print(f"Sampling {num_images} random images from the test set...")
    for i, batch in enumerate(loader):
        images, input_ids, masks, meta, labels = batch
        all_images.append(images)
        all_input_ids.append(input_ids)
        all_masks.append(masks)
        all_meta.append(meta)
        all_labels.append(labels)
        if i > 2: break

    all_images    = torch.cat(all_images)
    all_input_ids = torch.cat(all_input_ids)
    all_masks     = torch.cat(all_masks)
    all_meta      = torch.cat(all_meta)
    all_labels    = torch.cat(all_labels)

    indices = random.sample(range(len(all_images)), num_images)
    
    images         = all_images[indices].to(DEVICE)
    input_ids      = all_input_ids[indices].to(DEVICE)
    attention_mask = all_masks[indices].to(DEVICE)
    meta_data      = all_meta[indices].to(DEVICE)
    labels         = all_labels[indices].to(DEVICE)

    with torch.no_grad():
        preds, attn_weights = model(images, input_ids, attention_mask, meta_data, return_attn=True)
        
    preds = torch.argmax(preds, dim=1)
    
    images      = images.cpu()
    attn_weights = attn_weights.cpu()
    
    fig, axes = plt.subplots(5, 4, figsize=(20, 20))
    axes = axes.flatten()
    
    for i in range(num_images):
        img = images[i].permute(1, 2, 0).numpy()
        mean = np.array([0.485, 0.456, 0.406])
        std  = np.array([0.229, 0.224, 0.225])
        img  = np.clip(std * img + mean, 0, 1)
        
        true_label = class_names[labels[i].item()]
        pred_label = class_names[preds[i].item()]
        
        attn_map = attn_weights[i].mean(dim=0).reshape(8, 8).numpy()
        attn_map = cv2.resize(attn_map, (img.shape[1], img.shape[0]))
        attn_map = (attn_map - np.min(attn_map)) / (np.max(attn_map) + 1e-8)
        
        img_idx  = i * 2
        heat_idx = img_idx + 1
        
        axes[img_idx].imshow(img)
        axes[img_idx].set_title(f"True: {true_label}", fontsize=10)
        axes[img_idx].axis('off')
        
        axes[heat_idx].imshow(img)
        axes[heat_idx].imshow(attn_map, cmap='jet', alpha=0.5)
        axes[heat_idx].set_title(f"Pred: {pred_label}", fontsize=10)
        axes[heat_idx].axis('off')
        
    plt.tight_layout()
    plt.show()

visualize_attention_random(model, test_loader, class_names, num_images=10)

In [ ]:
def extract_fusion_features(model, loader, device):
    model.eval()
    all_features, all_labels = [], []

    with torch.no_grad():
        for images, input_ids, attention_mask, meta_data, labels in loader:
            images         = images.to(device)
            input_ids      = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            meta_data      = meta_data.to(device)

            # VISION
            f_convnext = model.convnext_features(images).flatten(2).transpose(1, 2)
            f_swin     = model.swin_norm(model.swin_features(images)).flatten(1, 2)
            fused_vision = model.vision_fuse(torch.cat([f_convnext, f_swin], dim=-1))

            # TEXT
            text_feats = model.text_encoder(
                input_ids=input_ids, attention_mask=attention_mask
            ).last_hidden_state

            # SHARED PROJECTION SPACE
            v_emb = F.normalize(model.vis_proj(fused_vision) + model.pos_embed, p=2, dim=-1)
            t_emb = F.normalize(model.text_proj(text_feats),                   p=2, dim=-1)

            # BIDIRECTIONAL CROSS-ATTENTION
            t_attended, _ = model.text_to_vis_attn(query=t_emb, key=v_emb, value=v_emb)
            v_attended, _ = model.vis_to_text_attn(query=v_emb, key=t_emb, value=t_emb)

            t_attended_pooled = t_attended.mean(dim=1)
            v_attended_pooled = v_attended.mean(dim=1)
            global_text       = t_emb.mean(dim=1)
            global_vision     = v_emb.mean(dim=1)

            # TABULAR
            f_meta      = model.tabular_mlp(meta_data)
            f_meta_proj = model.meta_proj(f_meta)

            # GATED FUSION -> FINAL 256-D EMBEDDING

            modalities   = [t_attended_pooled, v_attended_pooled, global_text, global_vision, f_meta_proj]
            stacked      = torch.stack(modalities, dim=1)
            gate_weights = model.modal_gate(
                torch.cat(modalities, dim=-1)
            ).softmax(dim=-1).unsqueeze(-1)
            fused        = (stacked * gate_weights).sum(dim=1)  # [B, 256]

            all_features.append(fused.cpu().numpy())
            all_labels.append(labels.numpy())

    all_features = np.concatenate(all_features, axis=0)
    all_labels   = np.concatenate(all_labels, axis=0)
    return all_features, all_labels

all_features, all_labels = extract_fusion_features(model, test_loader, DEVICE)

print("Feature Shape:", all_features.shape)
tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate=200,
    random_state=42,
    init='pca'
)

tsne_features = tsne.fit_transform(all_features)
plt.figure(figsize=(12, 10))

scatter = plt.scatter(
    tsne_features[:, 0],
    tsne_features[:, 1],
    c=all_labels,
    cmap='tab10',
    s=60,
    alpha=0.8
)

plt.legend(
    handles=scatter.legend_elements()[0],
    labels=class_names,
    title="Classes",
    fontsize=12
)

plt.title(
    "t-SNE Visualization of FusionNet Multimodal Embeddings over PAD-UFES-20 Test Set",
    fontsize=16
)

plt.xlabel("t-SNE Dimension 1", fontsize=14)
plt.ylabel("t-SNE Dimension 2", fontsize=14)

plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    "/kaggle/working/tsne_plot.pdf",
    dpi=600,
    bbox_inches='tight'
)

plt.show()